# 04 · Fixing the features and the data

Notebook 03 found two linked problems:

1. **Legit training hosts never have subdomains**, so the models learned "subdomain means phishing." On my history, model C flagged 84% of legit URLs with a subdomain and 2% without.
2. **63% of today's phishing lives on free hosting** (`*.webflow.io`, `*.workers.dev`), where the attacker picks the label in front of the platform. The subdomain shortcut happened to catch those, which inflated recall.

Two fixes, tested separately so we can tell which one helps:

| fix | what changes |
|---|---|
| **features (E)** | Parse hosts with the Public Suffix List. Replace "has a subdomain" with an explicit `is_free_hosting` flag, describe the label the site owner chose, and add a `brand_mismatch` check. |
| **data (+U)** | Add 50,000 legit *hostnames* from the Cisco Umbrella top list, which includes subdomains like `mail.yahoo.com`. |

That gives a 2x2 grid (C or E features, with or without Umbrella), plus the n-gram model D with and without Umbrella. Everything is scored on notebook 03's benchmark.

**Privacy and safety** are the same as notebook 03: history is shown only as totals unless `SHOW_PRIVATE = True`, and phishing URLs are shown defanged (`hxxps://evil[.]com`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

from phishdetect.data import load_phiusiil, PROJECT_ROOT
from phishdetect.features import host_only, split_host, featurize_host, HOST_FEATURE_NAMES
from phishdetect.models import build_models, host_psl_forest, HostCharNgram
from phishdetect.benchmark import load_tranco, load_openphish, load_chrome_history, load_umbrella, defang

pd.set_option("display.max_colwidth", 100)
RANDOM_STATE = 42
THRESHOLD = 0.5
SHOW_PRIVATE = False

## 1. The new host features
The Public Suffix List knows where the registered part of a hostname starts, including multi-part suffixes like `co.uk` and shared hosting like `webflow.io`. Each host gets split into:

- **sub**: everything in front of the registered domain (`docs` in `docs.google.com`)
- **owner**: the label the site's owner chose (`google`, or `gemini-login-b` in `gemini-login-b.webflow.io`)
- **suffix**: `com`, `co.uk`, `webflow.io`
- **platform**: whether it's shared hosting

The features then describe the owner label (length, digits, hyphens, randomness), the subdomain separately, and two flags: `is_free_hosting`, and `brand_mismatch`, which means a brand name appears in the host but isn't the owner label (`paypal.com.secure-verify.ru`).

In [ ]:
examples = ["https://docs.google.com/document/d/1", "https://www.amazon.com/dp/B0C",
            "https://gemini-login-b.webflow.io/", "http://abc123.gitbook.io/",
            "http://paypal.com.secure-verify.ru/login", "https://bbc.co.uk", "https://roblox.com.pt/games/1"]
parts = pd.DataFrame([split_host(u) for u in examples]).drop(columns="host")
feats = featurize_host(examples)[["n_sub_labels", "owner_len", "is_free_hosting", "brand_in_host", "brand_mismatch"]]
pd.concat([pd.Series([host_only(u) for u in examples], name="host"), parts, feats], axis=1)

Note the last row: `roblox.com.pt` gets `brand_mismatch = 0` because the owner label really is "roblox". A lookalike on a different country suffix fools this check. That's a known gap, and fixing it needs an allowlist of each brand's real domains.

## 2. Training data and benchmark
Same PhiUSIIL split and same benchmark sources as notebook 03.

In [ ]:
df = load_phiusiil().drop_duplicates("url").reset_index(drop=True)
y = df["is_phishing"].values
idx_train, idx_test = train_test_split(
    np.arange(len(df)), test_size=0.2, stratify=y, random_state=RANDOM_STATE)
train_urls, train_y = df["url"].values[idx_train], y[idx_train]

tranco  = load_tranco(n=5000, random_state=RANDOM_STATE)
phish   = load_openphish()
history = load_chrome_history(n=1000, max_per_host=3, random_state=RANDOM_STATE)
bench = pd.concat([tranco, history, phish], ignore_index=True)
bench["is_phishing"] = bench["is_phishing"].astype(int)
parts = pd.DataFrame([split_host(u) for u in bench["url"]])
bench["host"], bench["registered"] = parts["host"].values, parts["registered"].values
bench["platform"], bench["has_sub"] = parts["platform"].values, (parts["sub"] != "").values
bench.groupby("source").size().rename("urls")

## 3. Add Umbrella hostnames
Two safeguards:

- **Top 100k only.** The long tail of a DNS list is full of trackers and malware callbacks.
- **No benchmark domains.** Any Umbrella host whose registered domain appears anywhere in the benchmark is dropped. Otherwise the model could see `docs.google.com` in training and get "credit" for it in the benchmark. The model has to learn what legit subdomains look like in general, not memorize Google's.

In [ ]:
umbrella = load_umbrella(n=50_000, top_k=100_000, exclude_domains=set(bench["registered"]),
                         random_state=RANDOM_STATE)
umb_parts = pd.DataFrame([split_host(u) for u in umbrella["url"]])
print(f"{len(umbrella):,} Umbrella hosts added")
print(f"with a subdomain: {(umb_parts['sub'] != '').mean():.1%}")
print(f"on free hosting:  {umb_parts['platform'].mean():.1%}")
umbrella.sample(10, random_state=1)[["url", "rank"]]

In [ ]:
plus_urls = np.concatenate([train_urls, umbrella["url"].values])
plus_y = np.concatenate([train_y, np.zeros(len(umbrella), dtype=int)])
print(f"PhiUSIIL train: {len(train_urls):,} URLs, {train_y.mean():.1%} phishing")
print(f"plus Umbrella:  {len(plus_urls):,} URLs, {plus_y.mean():.1%} phishing")

# Sanity check: benchmark hosts that now appear in training (should be ~0 from Umbrella)
train_hosts_plus = set(host_only(u) for u in plus_urls)
bench["seen_in_train"] = bench["host"].isin(train_hosts_plus)
bench.groupby("source")["seen_in_train"].mean().round(3).rename("share seen in training")

## 4. Train the grid
Six models. C and D are the same as notebook 03 (their benchmark numbers should match); the rest are new.

In [ ]:
%%time
old = build_models(RANDOM_STATE)
grid = {
    "C · lexical":        (old["C · host only"],          train_urls, train_y),
    "C+U":                (build_models(RANDOM_STATE)["C · host only"], plus_urls, plus_y),
    "E · PSL":            (host_psl_forest(RANDOM_STATE), train_urls, train_y),
    "E+U":                (host_psl_forest(RANDOM_STATE), plus_urls, plus_y),
    "D · n-grams":        (HostCharNgram(),               train_urls, train_y),
    "D+U":                (HostCharNgram(),               plus_urls, plus_y),
}
models = {}
for name, (m, X, yy) in grid.items():
    models[name] = m.fit(X, yy)
    bench[f"p::{name}"] = m.phish_proba(bench["url"].values)
    print("trained", name)

## 5. Scoreboard
New columns compared with notebook 03:

- **Tranco FA** split into the top 10k (reliably legit) and the long tail (some sketchy sites, so partly label noise).
- **history FA (subdomain)**: the number this notebook is trying to fix.
- **recall (platform)** and **recall (other)**: OpenPhish recall split by whether the URL is on free hosting.
- **history vs phish PR-AUC**: ranking quality using only history (legit) and OpenPhish (phishing). It's the cleanest single number, since it skips Tranco's label noise.
- **history FA @ 90% recall**: pick, for each model, the threshold that catches 90% of OpenPhish, then report history false alarms at that threshold. Adding Umbrella shifts every model's probabilities, so a fixed 0.5 threshold isn't a fair comparison. This puts all models at the same recall. The threshold is tuned on the benchmark, so treat it as a comparison between models, not a promise.

In [ ]:
src = bench["source"]
is_t, is_h, is_p = src == "tranco", src == "history", src == "openphish"
top10k = bench["rank"] <= 10_000
hp = is_h | is_p

def rate(mask, flagged):
    return flagged[mask].mean() if mask.any() else np.nan

rows = []
for name in models:
    p = bench[f"p::{name}"]
    f = p >= THRESHOLD
    thr90 = np.quantile(p[is_p], 0.10)          # 90% of OpenPhish scores are at or above this
    rows.append({
        "model": name,
        "Tranco FA (top 10k)": rate(is_t & top10k, f),
        "Tranco FA (tail)": rate(is_t & ~top10k, f),
        "history FA": rate(is_h, f),
        "history FA (subdomain)": rate(is_h & bench["has_sub"], f),
        "recall": rate(is_p, f),
        "recall (platform)": rate(is_p & bench["platform"], f),
        "recall (other)": rate(is_p & ~bench["platform"], f),
        "history vs phish PR-AUC": average_precision_score(bench.loc[hp, "is_phishing"], p[hp]) if is_h.any() else np.nan,
        "history FA @ 90% recall": rate(is_h, p >= thr90),
    })
scoreboard = pd.DataFrame(rows).set_index("model").round(3)
scoreboard

In [ ]:
cols = ["history FA", "history FA (subdomain)", "Tranco FA (top 10k)", "recall"]
ax = scoreboard[cols].plot.barh(figsize=(9, 6))
ax.set_xlim(0, 1); ax.invert_yaxis(); ax.set_xlabel("rate (FA: lower is better, recall: higher is better)")
plt.tight_layout(); plt.show()

How to read it:

- **C vs E** (same data, different features) shows whether the PSL features alone fix the subdomain problem.
- **C vs C+U** and **E vs E+U** (same features, more data) show what Umbrella adds.
- **recall (platform) vs recall (other)**: if recall drops mostly on platform URLs, the old models were catching those through the subdomain shortcut.

## 6. What E+U learned

In [ ]:
models["E+U"].feature_importances().plot.barh(figsize=(7, 6), title="E+U feature importance"); plt.show()

## 7. Errors for the best model
Picked by history-vs-phish PR-AUC. Phishing URLs are defanged.

In [ ]:
best = scoreboard["history vs phish PR-AUC"].idxmax()
col = f"p::{best}"
print("best:", best)

print("\nTop-10k Tranco domains it flags most confidently:")
display(bench[is_t & top10k].nlargest(10, col)[["url", "rank", col]])

misses = bench[is_p].nsmallest(15, col)[["url", "platform", col]].copy()
misses["url"] = misses["url"].map(defang)
print("OpenPhish URLs it was most sure were safe:")
display(misses)

In [ ]:
h = bench[is_h]
if SHOW_PRIVATE:
    display(h.nlargest(20, col)[["host", "has_sub", col]])
else:
    print(f"{len(h)} history URLs; {best} flags {(h[col] >= THRESHOLD).sum()} at {THRESHOLD}.")
    print("Set SHOW_PRIVATE = True to see which hosts (and clear the output before committing).")

## 8. Save
The scoreboard (rates only, safe to commit) and the best model.

In [ ]:
import joblib
from datetime import date
(PROJECT_ROOT / "reports").mkdir(exist_ok=True)
scoreboard.assign(notebook="04", date=date.today().isoformat()).to_csv(PROJECT_ROOT / "reports" / "scoreboard_04.csv")
joblib.dump(models[best], PROJECT_ROOT / "models" / "host_v2.joblib")
print("saved", best)

## Results log
Fill in after running.

| model | history FA | history FA (subdomain) | recall | recall (platform) | hist vs phish PR-AUC | hist FA @ 90% recall |
|---|---|---|---|---|---|---|
| C · lexical | | | | | | |
| C+U | | | | | | |
| E · PSL | | | | | | |
| E+U | | | | | | |
| D · n-grams | | | | | | |
| D+U | | | | | | |

Questions for the journal:
1. Did the PSL features (C to E), the Umbrella data (C to C+U), or both fix the subdomain false alarms?
2. How much recall did we give up, and was it mostly on platform-hosted phishing?
3. Is the best model good enough to put behind an API yet? What's still missing?